## Link to article

This notebook is included in the documentation, where the interactive Plotly charts show up. See:
https://pegasystems.github.io/pega-datascientist-tools/latest/articles/vf_analysis.html

> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.


In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately

import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Value Finder Analysis
Every Value Finder simulation populates a dataset, the **pyValueFinder** dataset.  This dataset contains a lot more information than is what is currently presented on screen.

The data held in this dataset can be analysed to uncover insights into your decision framework. This notebook provides a sample analysis of the Value Finder simulation results.

In the data folder we've stored a copy of such a dataset, generated from an (internal) demo application (CDHSample). To run this notebook on your own data, you should export the **pyValueFinder** dataset from Dev Studio then follow the instructions below.

This is how Value Finder results of the sample data are presented in Pega (8.6, it may look different in other versions):

![Pega value finder screen](pegarun_8_6_0.png)

For the sample provided, the relevant action setting is 1.2%. There are 10.000 customers, 3491 without actions, 555 with only irrelevant actions and 5954 with at least one relevant action.

PDSTools defines a class **ValueFinder** that wraps the operations on this dataset. The "datasets" import is used for the example but you won't need this if you load your own Value Finder dataset.

Just like with the **ADMDatamart** class, you can supply your own path and filename as such:
```python
vf = ValueFinder(path = '[PATH TO DATA]', filename="[NAME OF DATASET EXPORT]")
```

- If only a path is supplied, it will automatically look for the latest file. 
- It is also possible to supply a dataframe as the 'df' argument directly, in which case it will use that instead. 


In [2]:
from pdstools import ValueFinder, datasets
import polars as pl
import numpy as np

# Load sample data - replace with your own data:
# vf = ValueFinder(path='...', filename='...')
vf = datasets.sample_value_finder()

When reading the data, we filter out unnecessary data, and the result is kept in the `df` property:

In [3]:
vf.df.head(5).collect()

ActionContext,Organization,TemplateName,Application,SubjectType,InteractionID,ApplicationVersion,DynamicTemplateIssue,SpecifySMSMessage,Rank,Segment,ModelExecutionError,DynamicTemplateGroup,Lastchannel,StartDate,Adminputsource,LastStage,ModelPropensity,Label,IsSMSEnabled,Unit,SMSMessage,UpdateopName,ContextWeight,UniqueID,Propensity,Interaction,AdjustedPropensityTreatment,Strategy,Value,SubjectID,Issue,Istransactional,KeyCode,ModelPerformance,Journey,BundleType,…,TreatmentKeyCode,Identifier,Bundleparent,OriginalSubjectID,StartingPropensity,ShortDescription,PreviousComponent,DynamicTemplateName,Operator,IsEmailEnabled,TemplateType,Stage,OptimizePredictionOutcome,Agentcompensation,StartingEvidence,Component,WorkID,Treatment,Prediction,PropensityThreshold,Division,Applyanalytics,Name,DecisionTime,AdjustedPropensityOmni,Benefits,ActionLimit,UpdateDateTime,Quantity,Direction,IsWebEnabled,BundleName,SpecifyEmailSubject,EmailSubject,Communicateto,IsPaidEnabled,Cost
str,str,str,str,cat,str,str,str,bool,u16,str,bool,str,str,str,str,str,f64,str,bool,str,str,str,str,str,f64,str,f64,str,f64,str,cat,bool,str,str,str,str,…,str,str,bool,str,str,str,str,str,str,bool,str,cat,bool,str,u32,str,str,str,str,f64,str,bool,str,datetime[ns],f64,str,i64,i64,i64,cat,bool,str,bool,str,str,bool,f64
"""Customer""","""DMOrg""","""""","""CDHSample""","""CDHSample-Data-Customer""","""6320879002230024255""","""01.01.01""","""Sales""",false,3,null,null,"""CreditCards""",null,"""20160201T163800.000 GMT""","""modelReferences""",null,0.269231,"""Student Checking""",true,"""Unit""","""""","""Piyush Vashisht""",null,null,0.269231,"""——""",0.278077,"""Opp_NBA_All""",null,"""Customer-1""","""Sales""",false,"""SDSC2""","""0.5""",null,"""FixedBundle""",…,"""""","""/Sales/DepositAccounts/Student…",false,"""Customer-1""","""0.5""","""Best of all, with Student Chec…","""Customers""","""ChannelAction_Template""","""Otto Perdeck""",true,"""""","""Applicability""",true,"""2""",100,"""Customers""","""Opp_NBA_AlDF_Simulat-1""","""Deposit_SMSTreatment""",null,0.0,"""Div""",true,"""StudentChecking""",2021-08-24 11:12:49.350,null,"""Students under age 23 are elig…",null,1629803581090,null,"""Outbound""",null,null,null,null,null,null,null
"""Customer""","""DMOrg""","""Mobile_SMSDBTemplate""","""CDHSample""","""CDHSample-Data-Customer""","""6320879002230024184""","""01.01.01""","""Usage""",true,3,null,null,"""Mobilebanking""",null,null,"""modelReferences""",null,0.5,"""Get the U+ Mobile App""",true,"""Unit""","""You are entitled for an offer""","""Piyush Vashisht""",null,null,0.5,"""——""",0.713095,"""Opp_NBA_All""",null,"""Customer-100""","""Usage""",false,"""UMGUMA1""","""0.5""",null,"""FixedBundle""",…,"""MOBILE""","""/Usage/Mobilebanking/GetTheUMo…",false,"""Customer-100""","""1""","""Manage accounts from your mobi…","""Customers""","""Usage_ChannelAction_Template""","""Otto Perdeck""",true,"""DB""","""Applicability""",true,null,25,"""Customers""","""Opp_NBA_AlDF_Simulat-1""","""Mobile_ExternalSMSTreatment""",null,0.0,"""Div""",true,"""GetTheUMobileApp""",2021-08-24 11:12:37.780,null,null,null,1629803581074,null,"""Outbound""",null,null,null,null,null,null,null
"""Customer""","""DMOrg""","""""","""CDHSample""","""CDHSample-Data-Customer""","""6320879002230023934""","""01.01.01""","""Collections""",false,3,null,null,"""Recommendations""",null,null,"""modelReferences""",null,0.5,"""Setup Autopay today""",true,"""Unit""","""""","""Piyush Vashisht""",null,null,0.5,"""——""",0.421306,"""Opp_NBA_All""",null,"""Customer-1000""","""Collections""",false,"""CRSAT1""","""0.5""",null,"""FixedBundle""",…,"""""","""/Collections/Recommendations/S…",false,"""Customer-1000""","""1""","""Setup autopay to avoid missing…","""Customers""","""Collections_ChannelAction_Temp…","""Otto Perdeck""",null,"""""","""Applicability""",true,null,25,"""Customers""","""Opp_NBA_AlDF_Simulat-1""","""Recommendations_SMSTreatment""",null,0.0,"""Div""",true,"""SetupAutopayToday""",2021-08-24 11:12:02.932,null,null,null,

## Basic Analysis: Pie Charts and Customer Segmentation

The piechart shown in platform is based on a propensity threshold. For the sample data, this threshold follows from a propensity quantile of 5.2%.

The [`plot.pie_charts()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.pie_charts) function shows piecharts for all stages in the engagement policies (in platform you only see the last one). The threshold is calculated automatically, but you can specify it explicitly.

In [4]:
# Automatic threshold calculation
vf.plot.pie_charts()

# Or specify threshold explicitly (e.g., 5.2% quantile)
vf.plot.pie_charts(quantiles=[0.052])

Hover over the charts to see the details. For the sample data, the rightmost pie chart corresponds to the numbers in Pega as shown in the screenshot above.

* Red = customers not receiving any action
* Yellow = customers not receiving any "relevant" actions, sometimes also called "under served"
* Green = customers that receive at least one "relevant" action, sometimes also called "well served"

With "relevant" defined as having a propensity above the threshold. This defaults to the 5th percentile.

## Propensity Analysis

Insights into the propensity distribution per stage are crucial. The [`plot.propensity_threshold()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.propensity_threshold) function visualizes this distribution. You often see a spike at 0.5, which corresponds to models without responses (their propensity defaults to 0.5/1 = 0.5).

The dotted vertical line represents the computed threshold.

In [5]:
vf.plot.propensity_threshold()

These different propensities represent 

* *pyModelPropensity* = the actual propensities from the models
* *pyPropensity* = model or random propensity, depending on the ModelControl (or, when models are executed from an extension point after the standard Predictions, their propensity, but such a configuration is not supported by Value Finder)
* *FinalPropensity* = the propensity after possible adjustments by Thompson Sampling; Thompson Sampling basically smoothes the propensities, you would expect any peak at 0.5 caused by empty models to be smoothed out

We can also look at the propensity distributions across the different stages. This is based on the model propensities, not any of the subsequent overrides:

In [6]:
vf.plot.propensity_distribution()

## Threshold Simulation

The effect of the propensity threshold selection on the number of actions for a customer can be simulated by supplying a list of quantiles or propensities. This generates aggregated counts per stage for different threshold values.

In [7]:
# Simulate thresholds from 1% to 99% in 1% increments
vf.plot.pie_charts(quantiles=np.arange(0.01, 1, 0.01))

The further to the left you put the slider threshold, the more "green" you will see. As you raise the threshold, more customers will be reported as getting "not relevant" actions.

The same effect can be visualized as a funnel. Use [`plot.distribution_per_threshold()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.distribution_per_threshold) to show the threshold on the x-axis.

In [8]:
# Default threshold range
vf.plot.distribution_per_threshold()

# Custom threshold range (1% to 99% in 1% increments)
vf.plot.distribution_per_threshold(quantiles=np.arange(0.01, 1, 0.01))

## Funnel Analysis

You can analyze how individual actions are distributed across stages. Since there are typically many actions, it's useful to filter to a specific group or issue.

The [`plot.funnel_chart()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.funnel_chart) function shows how actions flow through the stages. **Rule of thumb**: If there are only a few actions in each stage, or if certain actions are completely filtered out between stages, this may indicate strong filtering issues.

In [9]:
# Filter to Sales issue actions (adjust query for your data)
vf.plot.funnel_chart("Name", query=pl.col("Issue") == "Sales")

### Hierarchical Funnel Views

Start with a coarse-grained view at the Issue level, then drill down to Group or individual Action level as needed.

In [10]:
# High-level view: Issue level
vf.plot.funnel_chart("Issue")

# Mid-level view: Groups within Sales issue
vf.plot.funnel_chart("Group", query=pl.col("Issue") == "Sales")